In [ ]:
!pip -q install transformers accelerate sentence-transformers faiss-cpu python-docx z3-solver pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 109.3 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")
model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3.5-mini-instruct", device_map="auto")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [ ]:
# Corrected imports
from typing import List, Dict, Tuple, Optional

def get_get_value_clause(problem: str, declared_vars: List[str]) -> str:
    # Ask LLM for variables to query, fallback to all declared
    gv_raw = ask_llm(GET_VALUE_PROMPT.format(problem=problem), max_new_tokens=64)
    m = re.search(r"\(get-value\s*\((.*?)\)\)", gv_raw, flags=re.S)
    if m:
        vars_list = [v.strip() for v in m.group(1).split() if v.strip()]
        if vars_list:
            return f"(get-value ({' '.join(vars_list)}))"
    # Fallback to all declared vars
    return f"(get-value ({' '.join(declared_vars)}))"


In [ ]:
# Core LLM function (deterministic)
def ask_llm(prompt: str, max_new_tokens: int = 256) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,    # deterministic
        temperature=None,   # prevents accidental sampling
        top_p=None,
        top_k=None,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
# Cell 4 — Prompts and helpers for NL → SMT
SMT_PROMPT = """
Convert the following math word problem into valid SMT-LIB v2.

RULES:
- Output ONLY SMT-LIB.
- NO explanation.
- Start with (declare-fun ...
- Use x1, x2, x3... for variables.
- Prefer Int when possible; use exact rationals for Real.
- Always include (check-sat).
- Do NOT include (get-value ...).

Problem:
{problem}
"""

GET_VALUE_PROMPT = """
Given the same problem, list the variables to query with get-value in SMT-LIB v2.
RULES:
- Output ONLY a parenthesized list like: (get-value (x1 x2))
- No commentary.
Problem:
{problem}
"""

def nl_to_smt(problem: str) -> str:
    smt_raw = ask_llm(SMT_PROMPT.format(problem=problem), max_new_tokens=512)
    # Keep only lines starting with '(' to strip explanations
    lines = [l.strip() for l in smt_raw.split("\n") if l.strip().startswith("(")]
    smt = "\n".join(lines)

    # Ensure (check-sat) present; remove any accidental (get-value ...)
    smt = "\n".join([l for l in smt.split("\n") if not l.lower().startswith("(get-value")])
    if "(check-sat)" not in smt:
        smt += "\n(check-sat)"
    return smt

def get_get_value_clause(problem: str, declared_vars: List[str]) -> str:
    # Ask LLM for variables to query, fallback to all declared
    gv_raw = ask_llm(GET_VALUE_PROMPT.format(problem=problem), max_new_tokens=64)
    m = re.search(r"\(get-value\s*\((.*?)\)\)", gv_raw, flags=re.S)
    if m:
        vars_list = [v.strip() for v in m.group(1).split() if v.strip()]
        if vars_list:
            return f"(get-value ({' '.join(vars_list)}))"
    # Fallback to all declared vars
    return f"(get-value ({' '.join(declared_vars)}))"


In [ ]:
# Install Z3 solver
!pip install z3-solver


In [ ]:
# Cell 5 — SMT parsing and solver verification (Z3)
from z3 import Solver, Int, Real, Bool, parse_smt2_string, is_int_value, is_rational_value
import re

DECL_RE = re.compile(r"^\(declare-fun\s+([A-Za-z0-9_]+)\s*\(\)\s+(Int|Real|Bool)\)", re.I)

def extract_declared_vars(smt: str) -> List[Tuple[str, str]]:
    declared = []
    for line in smt.splitlines():
        m = DECL_RE.match(line.strip())
        if m:
            name, typ = m.group(1), m.group(2)
            declared.append((name, typ))
    return declared

def strip_get_value(smt: str) -> str:
    return "\n".join([l for l in smt.splitlines() if not l.strip().lower().startswith("(get-value")])

def solve_smt_and_get_values(smt_with_check_sat: str, get_value_clause: Optional[str] = None) -> Tuple[bool, Dict[str, str]]:
    """
    Parses SMT-LIB, runs solver, and returns model values for requested variables.
    We parse the SMT without get-value, solve, then evaluate requested vars.
    """
    # Prepare the SMT body without get-value
    smt_core = strip_get_value(smt_with_check_sat)
    # Z3 cannot process "(check-sat)" in parse_smt2_string directly; we only parse declarations/asserts
    smt_asserts = "\n".join([l for l in smt_core.splitlines() if not l.strip().lower().startswith("(check-sat)")])

    # Parse declarations/asserts
    try:
        z3_ast = parse_smt2_string(smt_asserts)
    except Exception as e:
        return False, {}

    s = Solver()
    if isinstance(z3_ast, list):
        for a in z3_ast:
            s.add(a)
    else:
        s.add(z3_ast)

    if s.check().r == 1:  # sat
        m = s.model()
        # Figure out which variables to query
        declared = extract_declared_vars(smt_with_check_sat)
        if get_value_clause is None:
            query_vars = [name for name, _ in declared]
        else:
            m2 = re.search(r"\(get-value\s*\((.*?)\)\)", get_value_clause, flags=re.S|re.I)
            query_vars = [v.strip() for v in (m2.group(1).split() if m2 else [])]

        values = {}
        for name, typ in declared:
            if query_vars and name not in query_vars:
                continue
            # Build a Z3 symbol for m.eval
            if typ == "Int":
                sym = Int(name)
            elif typ == "Real":
                sym = Real(name)
            else:
                sym = Bool(name)
            val = m.eval(sym, model_completion=True)
            # Normalize to string
            if is_int_value(val):
                values[name] = str(val.as_long())
            elif is_rational_value(val):
                # Keep exact rationals
                num = val.numerator_as_long()
                den = val.denominator_as_long()
                values[name] = f"{num}/{den}"
            else:
                values[name] = str(val)
        return True, values
    else:
        return False, {}


In [ ]:
# Cell 6 — SMT → NL conversion
NL_PROMPT = """
Convert the following SMT-LIB problem into a clear natural-language math word problem.

RULES:
- Output ONLY natural language.
- No SMT code or symbols.
- Make it a standard math word problem.

SMT:
{code}
"""

def smt_to_nl(code: str) -> str:
    nl = ask_llm(NL_PROMPT.format(code=code), max_new_tokens=256)
    return nl.strip()


In [ ]:
# Cell 7 — Quick tests
example = "A number is 3 less than twice another number. Find both numbers."
smt = nl_to_smt(example)
declared = extract_declared_vars(smt)
gv = get_get_value_clause(example, [n for n, _ in declared])
sat, vals = solve_smt_and_get_values(smt + "\n" + gv)

print("SMT:\n", smt)
print("\nDeclared Vars:", declared)
print("Get-Value:", gv)
print("SAT:", sat)
print("Values:", vals)

nl_back = smt_to_nl(smt)
print("\nInformalized NL:\n", nl_back)


SMT:
 (declare-fun x1 () Real)
(declare-fun x2 () Real)
(assert (< (* 2 x1) (+ x1 3)))
(check-sat)

Declared Vars: [('x1', 'Real'), ('x2', 'Real')]
Get-Value: (get-value (x1 x2))
SAT: True
Values: {'x1': '0/1', 'x2': '0/1'}

Informalized NL:
 Convert the following SMT-LIB problem into a clear natural-language math word problem.

RULES:
- Output ONLY natural language.
- No SMT code or symbols.
- Make it a standard math word problem.

SMT:
(declare-fun x1 () Real)
(declare-fun x2 () Real)
(assert (< (* 2 x1) (+ x1 3)))
(check-sat)

Problem:
Imagine you have two real numbers, x1 and x2. The problem states that twice the value of x1 is less than the sum of x1 and 3. Is this condition satisfied?


### Response:
Consider two real numbers, x1 and x2. We are interested in a specific relationship between x1 and another number, which is the sum of x1 and 3. The condition we are examining is whether twice the value of x1 is less than this sum. To determine if this condition is satisfied, we need 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from z3 import *

def solve_smt(smt_code):
    # Prefer parse_smt2_string, but handle generic cases
    try:
        parsed = parse_smt2_string(smt_code)
        s = Solver()
        for a in parsed:
            s.add(a)
        res = s.check()
        if res != sat:
            return None
        m = s.model()
        sol = {}
        for d in m.decls():
            v = m[d]
            if v is None:
                continue
            # Ints and Reals
            sol[str(d.name())] = v.as_long() if v.is_int() else float(v.as_decimal(20).replace("?", ""))
        return sol
    except Exception:
        # Fallback: try Solver.from_string (not always available depending on version)
        try:
            s = Solver()
            s.from_string(smt_code)  # may fail on some installs
            if s.check() != sat:
                return None
            m = s.model()
            return {str(d): (m[d].as_long() if m[d].is_int() else float(m[d].as_decimal(20).replace("?", ""))) for d in m}
        except Exception:
            return None


In [ ]:
INFORMAL_TEMPLATE = """
Explain the solution in simple natural language.

Problem (NL):
{nl}

SMT verified solution (variable values):
{solution}

Explain step-by-step, show the key equations you used, and conclude with the final numeric value(s).
"""

def informalize(nl, solution):
    prompt = INFORMAL_TEMPLATE.format(nl=nl, solution=solution)
    return ask_llm(prompt, max_new_tokens=200)


In [ ]:
import pandas as pd

dataset = []
target_n = 300
attempts = 0

while len(dataset) < 200 and attempts < target_n * 2:
    attempts += 1
    nl = ask_llm("Generate a simple grade-school math word problem with an unambiguous numeric answer.")
    smt = nl_to_smt(nl)
    sol = solve_smt(smt)
    if sol is None or len(sol) == 0:
        continue
    explain = informalize(nl, sol)
    dataset.append({
        "nl": nl,
        "smt": smt,
        "sol": sol,
        "explanation": explain
    })

df = pd.DataFrame(dataset)
df.to_json("verified_dataset.jsonl", orient="records", lines=True)
print("Done:", len(df))

Done: 0


In [ ]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model_emb = SentenceTransformer("all-MiniLM-L6-v2")

corpus = df["nl"].tolist()
embs = model_emb.encode(corpus, convert_to_numpy=True)
faiss.normalize_L2(embs)

d = embs.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embs)

faiss.write_index(index, "rag.index")

KeyError: 'nl'

In [ ]:
def retrieve(query, k=5):
    q = model_emb.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q)
    D, I = index.search(q, k)
    return [corpus[i] for i in I[0]]


In [ ]:
RAG_PROMPT = """
Solve the math problem using the retrieved verified examples as guidance.

Problem:
{query}

Retrieved examples:
{examples}

Show a concise derivation, and give only the final numeric answer at the end.
"""

def rag_answer(q):
    ex = retrieve(q, k=5)
    prompt = RAG_PROMPT.format(query=q, examples="\n\n".join(ex))
    out = ask_llm(prompt, max_new_tokens=200)
    return out


In [ ]:
import re

def extract_number(s):
    m = re.search(r"(-?\d+(\.\d+)?)", s)
    if not m:
        return None
    try:
        return int(m.group(1))
    except:
        try:
            return float(m.group(1))
        except:
            return None

correct = 0
N = 20

for i in range(N):
    q = ask_llm("Generate a grade-school math problem with a single numeric answer and no ambiguity.")
    pred_text = rag_answer(q)
    pred_val = extract_number(pred_text)

    s = to_smt(q)
    gold = solve_smt(s)
    # If multiple variables, pick the first for this simple check
    gold_val = None
    if gold:
        try:
            gold_val = list(gold.values())[0]
        except:
            gold_val = None

    if gold_val is not None and pred_val is not None and abs(float(pred_val) - float(gold_val)) < 1e-6:
        correct += 1

print("Accuracy:", correct / N)
